# ROS Assay Analysis
This notebook processes raw luminescence data from ROS assays, checks for statistical normality among sample groups, and generates a grouped boxplot with overlaid strip plots and Compact Letter Display (CLD) statistical annotations. Run the first cells for imports

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.lines as mlines
from matplotlib.patches import Patch
import scipy.stats as stats

%matplotlib inline

# Helper functions

In [ ]:
# ---------- Helper: Process Plate Data ----------
def process_plate_data(file_path, treatment_map, exp_label):
    try:
        df = pd.read_excel(file_path)
    except FileNotFoundError:
        print(f"File not found: {file_path}. Please check the path.")
        return pd.DataFrame()

    sum_rows = []
    for treatment, wells in treatment_map.items():
        for w in wells:
            if w in df.columns:
                total = pd.to_numeric(df[w], errors='coerce').sum(skipna=True, min_count=1)
                sum_rows.append({
                    'Treatment': treatment, 
                    'Well': w, 
                    'Sum_RLU': total,
                    'Experiment': exp_label
                })
    
    sums_df = pd.DataFrame(sum_rows).dropna(subset=['Sum_RLU']).copy()
    sums_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    sums_df.dropna(subset=['Sum_RLU'], inplace=True)
    return sums_df

# Defining treatments and filenames

In [ ]:
# ---------- Define Layouts & Load Data ----------
all_rows = 'ABCDEFGH'

# Make sure to update these paths if running on a different machine
file_exp1 = 'Exp1.xlsx'
map_exp1 = {
    'Water/Water':      [f'{r}1' for r in all_rows],
    'Water/flg22':      [f'{r}2' for r in all_rows],
    'Xtu-WT/Water':     [f'{r}5' for r in all_rows], 
    'Xtu-WT/flg22':     [f'{r}6' for r in all_rows], 
    'Xtu-GumD/Water':   [f'{r}7' for r in all_rows], 
    'Xtu-GumD/flg22':   [f'{r}8' for r in all_rows], 
    'Xtu-HrcT/Water':   [f'{r}9' for r in all_rows], 
    'Xtu-HrcT/flg22':   [f'{r}10' for r in all_rows],
}

file_exp2 = 'Exp2.xlsx'
map_exp2 = {
    'Water/Water':      [f'{r}1' for r in all_rows],
    'Water/flg22':      [f'{r}2' for r in all_rows],
    'Xtu-WT/Water':     [f'{r}3' for r in all_rows],
    'Xtu-WT/flg22':     [f'{r}4' for r in all_rows],
    'Xtu-GumD/Water':   [f'{r}5' for r in all_rows],
    'Xtu-GumD/flg22':   [f'{r}6' for r in all_rows],
    'Xtu-HrcT/Water':   [f'{r}7' for r in all_rows],
    'Xtu-HrcT/flg22':   [f'{r}8' for r in all_rows],
}

# Process and Combine
df1 = process_plate_data(file_exp1, map_exp1, 'Exp 1')
df2 = process_plate_data(file_exp2, map_exp2, 'Exp 2')
combined_df = pd.concat([df1, df2], ignore_index=True)

# Define Base Treatment Order
treatment_order = [
    'Water/Water', 'Water/flg22', 
    'Xtu-GumD/Water', 'Xtu-GumD/flg22',
    'Xtu-HrcT/Water', 'Xtu-HrcT/flg22', 
    'Xtu-WT/Water', 'Xtu-WT/flg22'
]
treatment_order = [t for t in treatment_order if t in combined_df['Treatment'].unique()]
combined_df['Treatment'] = pd.Categorical(combined_df['Treatment'], categories=treatment_order, ordered=True)

# Show a preview of the data
display(combined_df.head())

In [ ]:
# ==========================================================
# NORMALITY TESTING (Shapiro-Wilk)
# ==========================================================
normality_results = []

for treatment, group in combined_df.groupby('Treatment', observed=False):
    data = group['Sum_RLU'].dropna()
    
    # Shapiro-Wilk requires at least 3 data points
    if len(data) >= 3:
        stat, p_value = stats.shapiro(data)
        is_normal = p_value > 0.05
    else:
        stat, p_value = np.nan, np.nan
        is_normal = "Not enough data"
        
    normality_results.append({
        'Treatment': treatment,
        'N (Sample Size)': len(data),
        'W-Statistic': stat,
        'p-value': p_value,
        'Is Normal (α=0.05)': is_normal
    })

normality_df = pd.DataFrame(normality_results)

# Display the results cleanly in the notebook
display(normality_df)

## Data Visualization
Generating a grouped boxplot overlaid with independent sample points from multiple experimental runs.

In [ ]:
# ==========================================================
# AUTOMATED STATS & CLD MAP GENERATION
# ==========================================================
import scipy.stats as stats
import scikit_posthocs as sp
import networkx as nx
import string

print("Evaluating normality to select statistical test...")
valid_df = combined_df.dropna(subset=['Sum_RLU'])
groups_dict = {trt: group['Sum_RLU'].values for trt, group in valid_df.groupby('Treatment', observed=False) if len(group) > 0}
groups_list = list(groups_dict.values())

# 1. Check Normality (Shapiro-Wilk)
is_normal = True
for trt, data in groups_dict.items():
    if len(data) >= 3:
        stat, p = stats.shapiro(data)
        if p < 0.05:
            is_normal = False
            break
    else:
        is_normal = False # Not enough data in a group to assume normality
        break

cld_map = {}
stat_name = ""

# 2. Perform Appropriate Test
if is_normal:
    stat_name = "Parametric (ANOVA / Tukey's HSD)"
    print(f"Data appears normal. Running {stat_name}...")
    stat_val, p_val = stats.f_oneway(*groups_list)
    
    if p_val < 0.05:
        p_matrix = sp.posthoc_tukey(valid_df, val_col='Sum_RLU', group_col='Treatment')
else:
    stat_name = "Non-Parametric (Kruskal-Wallis / Dunn's)"
    print(f"Data violates normality. Running {stat_name}...")
    stat_val, p_val = stats.kruskal(*groups_list)
    
    if p_val < 0.05:
        p_matrix = sp.posthoc_dunn(valid_df, val_col='Sum_RLU', group_col='Treatment', p_adjust='fdr_bh')

# 3. Generate Letters (CLD)
if p_val < 0.05:
    G = nx.Graph()
    G.add_nodes_from(p_matrix.columns)
    
    # Add an edge if the difference is NOT significant (p >= 0.05)
    for i, trt1 in enumerate(p_matrix.columns):
        for j, trt2 in enumerate(p_matrix.columns):
            if i < j and p_matrix.loc[trt1, trt2] >= 0.05:
                G.add_edge(trt1, trt2)
                
    # Find overlapping groups (cliques)
    cliques = list(nx.find_cliques(G))
    
    # Sort cliques so highest median gets the letter 'a'
    medians = valid_df.groupby('Treatment', observed=False)['Sum_RLU'].median()
    cliques.sort(key=lambda c: max([medians.get(node, 0) for node in c]), reverse=True)
    
    # Assign letters to groups
    letters = string.ascii_lowercase
    for node in G.nodes(): cld_map[node] = ""
    for i, clique in enumerate(cliques):
        letter = letters[i % 26]
        for node in clique: cld_map[node] += letter
            
    # Alphabetize letters for clean plotting
    for node in cld_map: cld_map[node] = "".join(sorted(list(set(cld_map[node]))))
else:
    print(f"Overall test not significant (p={p_val:.4f}). Assigning 'a' to all.")
    cld_map = {t: 'a' for t in groups_dict.keys()}

print("Assigned Letters:", cld_map)

# ==========================================================
# FORMAT FOR GROUPED PLOTTING
# ==========================================================
combined_df[['Strain', 'Infiltration']] = combined_df['Treatment'].str.split('/', n=1, expand=True)

strain_map = {
    'Water': 'Water',
    'Xtu-WT': 'Xtu-WT',
    'Xtu-GumD': 'Xtu-ΔGumD',
    'Xtu-HrcT': 'Xtu-ΔHrcT'
}
combined_df['Strain_Label'] = combined_df['Strain'].map(strain_map)

# Define requested order
strain_order = ['Water', 'Xtu-WT', 'Xtu-ΔGumD', 'Xtu-ΔHrcT']
inf_order = ['Water', 'flg22']

# ==========================================================
# PLOT GENERATION
# ==========================================================
fig, ax = plt.subplots(figsize=(14, 7))

# Style settings
box_kw    = dict(edgecolor='black', linewidth=1.0)
line_kw   = dict(color='black',  linewidth=1.0)
median_kw = dict(color='black',  linewidth=1.2)

# Boxplot grouped by Strain (X) and Infiltration (Hue)
sns.boxplot(
    data=combined_df, x='Strain_Label', y='Sum_RLU', hue='Infiltration',
    order=strain_order, hue_order=inf_order, width=0.7, showfliers=False, ax=ax,
    palette=['whitesmoke', 'lightgrey'], 
    boxprops=box_kw, whiskerprops=line_kw, capprops=line_kw, medianprops=median_kw
)

# Scatter points dodged perfectly over the boxes
exp_styles = {
    'Exp 1': {'marker': 'o', 'color': 'black'},
    'Exp 2': {'marker': '^', 'color': 'dimgrey'}
}

for exp_label, style in exp_styles.items():
    subset = combined_df[combined_df['Experiment'] == exp_label]
    if subset.empty: continue
    
    sns.stripplot(
        data=subset, x='Strain_Label', y='Sum_RLU', hue='Infiltration',
        order=strain_order, hue_order=inf_order, dodge=True,
        marker=style['marker'], alpha=0.85, s=7, 
        edgecolor='white', linewidth=0.5, jitter=True, ax=ax, legend=False, zorder=10,
        palette=[style['color'], style['color']] 
    )

# ----- Add CLD letters above the grouped box plots -----
max_values = combined_df.groupby('Treatment', observed=False)['Sum_RLU'].max()
y_range = combined_df['Sum_RLU'].max() - combined_df['Sum_RLU'].min()
y_offset = y_range * 0.05

for i, strain_label in enumerate(strain_order):
    for j, inf in enumerate(inf_order):
        raw_strain = [k for k, v in strain_map.items() if v == strain_label][0]
        treatment_key = f"{raw_strain}/{inf}"
        
        if treatment_key in cld_map and treatment_key in max_values.index:
            letter = cld_map[treatment_key]
            y_pos = max_values.loc[treatment_key] + y_offset
            
            x_pos = i - 0.2 if inf == 'Water' else i + 0.2
            ax.text(x_pos, y_pos, letter, ha='center', va='bottom', 
                    fontsize=14, fontweight='bold', color='darkred')

y0, y1_curr = ax.get_ylim()
top_needed = combined_df['Sum_RLU'].max() + (y_offset * 3)
if top_needed > y1_curr:
    ax.set_ylim(y0, top_needed)

ax.set_xlabel('Background Strain', fontsize=12)
ax.set_ylabel('Total ROS (sum of RLU over time)', fontsize=12)
ax.set_title(f'ROS Assay: Total RLU per Well\nStats: {stat_name}', pad=12)

inf_handles = [Patch(facecolor='whitesmoke', edgecolor='black', label='Water'),
               Patch(facecolor='lightgrey', edgecolor='black', label='flg22')]
leg1 = ax.legend(handles=inf_handles, title='Infiltration', loc='upper left', bbox_to_anchor=(1.02, 1))
ax.add_artist(leg1)

exp_handles = [mlines.Line2D([], [], color='white', marker=style['marker'], 
                             markerfacecolor=style['color'], markeredgecolor='white', 
                             markersize=8, label=exp) for exp, style in exp_styles.items()]
ax.legend(handles=exp_handles, title='Plate Run', loc='upper left', bbox_to_anchor=(1.02, 0.8))

fig.tight_layout()

fig.savefig('ROS_grouped_sum_RLU.svg', bbox_inches='tight')
plt.show()